# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.
We will list all record sets and their fields present in the dataset using their `@id` values.

In [ ]:
# List all record sets with their @id
record_sets = list(dataset.record_sets)
print(f"Record sets found ({len(record_sets)}):")
for rs in record_sets:
    print(f"- Record Set: {rs['@id']}")
    # List the fields and their @id, if present
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        print(f"    - Field: {f['@id'] if isinstance(f, dict) and '@id' in f else f}")

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis.
Below, we load all tabular record sets and preview their fields and first records. All lookups and assignments use the `@id`.

**Note:** Set the desired record set `@id` values to extract.

In [ ]:
# Extract data from each record set
# Identify all record sets (using @id)
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
# Load up to 1000 records per set for preview
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"DataFrame for {record_set_id} columns: {df.columns.tolist()}")
            print(f"First 3 records from {record_set_id}:")
            display(df.head(3))
        else:
            print(f"No records found in: {record_set_id}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

# Choose one record set to use for EDA, e.g. the first non-empty
main_record_set_id = next((rsid for rsid in dataframes if not dataframes[rsid].empty), None)
print(f"Using record set: {main_record_set_id} for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

All columns are referenced by their field `@id`, as shown in the overview above.

In [ ]:
from pandas.api.types import is_numeric_dtype

# Select a numeric field for analysis (first numeric column, using @id)
df = dataframes[main_record_set_id]
numeric_field_id = None
for col in df.columns:
    if is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise ValueError('No numeric fields found in the selected record set.')

print(f"Selected numeric field: {numeric_field_id}")

# Example threshold (10) for demonstration. Adjust as relevant for specific field.
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Try grouping by the first categorical column
group_field = None
for col in df.columns:
    if df[col].dtype == object and col != numeric_field_id:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field}:")
    display(grouped_df.head())
else:
    print("No categorical grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field after filtering
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id} (>{threshold}) in {main_record_set_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Boxplot of numeric field grouped by a categorical field, if available
if group_field:
    plt.figure(figsize=(12,5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded metadata and records using `mlcroissant`, referencing all fields and record sets by their `@id` where possible.
- Explored records and fields in the dataset, and extracted tabular data for further analysis.
- Demonstrated filtering, normalization, and grouping by field, as well as data visualization using common Python tools.
- With the power of Croissant schemas and the `mlcroissant` library, reproducible FAIR dataset analysis is streamlined.
